# EAGLE workflow

Prepare datasets and networks, train the three source models, and run the six evaluations. Open the notebook from the repository root and select the **EAGLE** dependency environment described in `README.md`. All script calls use `python -m EAGLE <command>` through the current kernel interpreter.

Start Ryu and Mininet in external terminals with the selected network, starting Ryu first. Restart both whenever changing networks. Collect each topology in a separate invocation of the collection cell.


In [ ]:
import json
import shlex
import subprocess
import sys

from IPython.display import SVG, Image, display

from EAGLE.src import config
from EAGLE.src.paths import ROOT
from network_schema import resolve_support_counts

PYTHON = sys.executable
SOURCE_HOSTS = 9
V4_HOSTS = 30
SEED = config.SEED
MIN_DECISIONS = config.MIN_DECISIONS
SKIP_PCAPS = False
SKIP_V4_TRAIN = True
DECISION_SECONDS = config.DECISION_SECONDS
SUPPORT_SWITCHES = resolve_support_counts(config.SWITCHES_PER_DOMAIN, config.N_DOMAINS)
EPISODES = 15
PRETRAIN_EPISODES = 5
POSTTRAIN_EPISODES = 5
EVAL_EPISODES = 5
REPETITIONS = 1
SOURCE_MODELS = ["dc_t0", "dc_pcaps", "dc_full"]
MODELS = SOURCE_MODELS.copy()
SOURCE_DATA = ROOT / f"datasets/{SOURCE_HOSTS}_hosts"
V4_DATA = ROOT / f"datasets/{V4_HOSTS}_hosts"


def run(command_name, *arguments):
    """Stream a source command and stop its child when the cell is interrupted."""
    command = [PYTHON, "-m", "EAGLE", command_name, *map(str, arguments)]
    print(shlex.join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if status:
        raise subprocess.CalledProcessError(status, command)


def show_png(path):
    display(Image(filename=str(ROOT / path)))


def show_network(name):
    display(SVG(filename=str(ROOT / f"networks/{name}.svg")))

## Inspect source captures

The inputs are `5gb-tcp-connection.pcap.gz` and Megalodon's Content Server capture saved as `megalodon.pcapng` in the repository root. These cells analyze complete captures and display their throughput histories.


In [ ]:
run("analyze-trace")
run("analyze-trace", "--mg")
show_png("datasets/analyses/dc/traffic_1s.png")
show_png("datasets/analyses/mg/traffic_1s.png")

## Prepare source and larger-host datasets

Each command creates a fresh host-count directory. Move existing output aside before regenerating it. Pooling uses every available split window before repeating. `min_decisions=20` requires at least 100 seconds per direction after scaling at five seconds per decision. Source windows are repeated only when their scaled duration falls short of this minimum. Megalodon synthetic traffic also supplies test condition T2.

`config.TRAFFIC_SCALE_FACTOR` in `EAGLE.src.config` controls time stretching. `None` uses the host count. With nine hosts, 12 source windows produce 108 seconds and 21 complete five-second decisions, meeting a minimum of 20. Longer available sequences are retained. Synthetic traffic preserves each window's bytes over its longer delivery interval.

Set `SKIP_PCAPS=True` for synthetic-only generation without packet spooling. `SKIP_V4_TRAIN=True` generates only test datasets for the larger network. PCAP training and evaluation cells use captures from a full preparation run.


In [ ]:
for hosts in (SOURCE_HOSTS, V4_HOSTS):
    options = []
    if SKIP_PCAPS:
        options.append("--skip-pcaps")
    if hosts == V4_HOSTS and SKIP_V4_TRAIN:
        options.append("--skip-train")
    run(
        "prepare-data",
        "default",
        "--hosts",
        hosts,
        "--min-decisions",
        MIN_DECISIONS,
        "--decision-seconds",
        DECISION_SECONDS,
        "--seed",
        SEED,
        *options,
    )

manifest_names = (
    ("dc_t0", "mg_t0") if SKIP_PCAPS else ("dc_t0", "dc_pcap", "mg_t0", "mg_pcap")
)
for name in manifest_names:
    manifest = json.loads((SOURCE_DATA / "test" / f"{name}.json").read_text())
    print(name, {key: manifest[key] for key in ("duration_seconds", "decisions")})

In [ ]:
show_png(SOURCE_DATA / "test/dc_t0.png")
show_png(SOURCE_DATA / "test/mg_t0.png")

## Generate topology files

V0 preserves the source switch graph and takes host identities from the prepared dataset. V3 extends V0 to 16 switches and 22 inter-switch links. V1 and V2 use the random generator. V4 uses GtsCe with the larger-host dataset. Adjust the random topology inputs and V4 host count above for your experiment.


In [ ]:
source_manifest = SOURCE_DATA / "train/dc_t0.json"
run(
    "prepare-network",
    "v0",
    "--dataset",
    source_manifest,
    "--seed",
    SEED,
    "--support-switches",
    *SUPPORT_SWITCHES,
)
run(
    "prepare-network",
    "v3",
    "--dataset",
    source_manifest,
    "--seed",
    SEED,
    "--support-switches",
    *SUPPORT_SWITCHES,
)
run(
    "prepare-network",
    "random",
    "--nodes",
    13,
    "--links",
    16,
    "--dataset",
    source_manifest,
    "--name",
    "v1",
    "--seed",
    SEED,
    "--support-switches",
    *SUPPORT_SWITCHES,
)
run(
    "prepare-network",
    "random",
    "--nodes",
    13,
    "--links",
    18,
    "--dataset",
    source_manifest,
    "--name",
    "v2",
    "--seed",
    SEED,
    "--support-switches",
    *SUPPORT_SWITCHES,
)
run(
    "prepare-network",
    "zoo",
    "--topo",
    "GtsCe",
    "--dataset",
    V4_DATA / "test/dc_t0.json",
    "--name",
    "v4",
    "--seed",
    SEED,
    "--support-switches",
    *SUPPORT_SWITCHES,
)

In [ ]:
show_network("v0")
show_network("v1")
show_network("v2")
show_network("v3")
show_network("v4")

## Prepare curriculum congestion environments

TrEnv 1 uses DC T0. TrEnvs 2 to 4 increase traffic from one controller domain, while TrEnv 5 increases traffic from all domains.


In [ ]:
run(
    "prepare-data",
    "congestion",
    "--hosts",
    SOURCE_HOSTS,
    "--network",
    ROOT / "networks/v0.json",
)
# show_png(SOURCE_DATA / "train/dc_t0_trenv2.png")
# show_png(SOURCE_DATA / "train/dc_t0_trenv5.png")

## Start the V0 services externally

Run these commands in separate terminals, Ryu first.

```sh
make controllers ARGS='--network networks/v0.json'
make network ARGS='--network networks/v0.json'
```

The following cells discover their socket addresses. They assume both services use V0. Every model calibrates on its selected training conditions, pretrains its reward estimator, and freezes that estimator during actor–critic learning. The policy episode budget excludes calibration and pretraining.


In [ ]:
for traffic, name in zip(("synthetic", "pcaps", "curriculum"), SOURCE_MODELS):
    run(
        "train",
        "--network",
        "v0.json",
        "--traffic",
        traffic,
        "--output",
        name,
        "--episodes",
        EPISODES,
        "--pretrain-episodes",
        PRETRAIN_EPISODES,
        "--seed",
        SEED,
    )
    show_png(f"models/{name}.png")

To resume an interrupted run, repeat its original command with `--resume`. Each named inference file is accompanied by its CSV/PNG. The latest resumable state and event log are under `models/checkpoints/`.


## Post-train all three models on DC T5

Generate the reduced-load T5 manifests with the scaling command. Post-training uses the training split and saves each adapted model under a new name.


In [ ]:
for split in ("train", "test"):
    run(
        "scale-traffic",
        SOURCE_DATA / split / "dc_t0.json",
        SOURCE_DATA / split / "dc_t5.json",
        "--factor",
        0.2,
    )

POSTTRAIN_MODELS = [f"{name}_t5" for name in SOURCE_MODELS]
for source, name in zip(SOURCE_MODELS, POSTTRAIN_MODELS):
    options = ["--resume"] if (ROOT / f"models/checkpoints/{name}.pth").exists() else []
    run(
        "train",
        "--network",
        "v0.json",
        "--dataset",
        "dc_t5",
        "--from-checkpoint",
        source,
        "--output",
        name,
        "--episodes",
        POSTTRAIN_EPISODES,
        "--seed",
        SEED,
        *options,
    )
    show_png(f"models/{name}.png")
MODELS = POSTTRAIN_MODELS.copy()

## Train pruning variants

Use the same synthetic source and episode budget. The default `dc_t0` model provides dynamic pruning. These two additional models provide the dense and static-pruning reward curves.


In [ ]:
for name, settings in (
    ("dc_t0_no_pruning", "settings/no_pruning.json"),
    ("dc_t0_static_pruning", "settings/static_pruning.json"),
):
    run(
        "train",
        "--network",
        "v0.json",
        "--traffic",
        "synthetic",
        "--output",
        name,
        "--settings",
        settings,
        "--episodes",
        EPISODES,
        "--pretrain-episodes",
        PRETRAIN_EPISODES,
        "--seed",
        SEED,
    )
    show_png(f"models/{name}.png")

## Evaluation 2 — traffic embeddings on V0

Keep V0 running. Collect one embedding at the first post-reset observation with delivered traffic from T0, T1, Megalodon T2, T3, T4, and TrEnvs 2 to 5. All switches stay on during collection. All nine vectors are projected together.


In [ ]:
run("eval-traffic-embeddings", "--model", MODELS[0], "--seed", SEED)
show_png("eval/traffic_embeddings.png")

## Evaluation 3 — model comparison across datasets

Keep V0 running. Each repetition uses `SEED + i` for all three models and the matched all-on baseline. The source dataset group contains the two synthetic source datasets. The load group contains T1, T3, and T4, with Megalodon T2 already included among the source datasets. The congestion group contains TrEnvs 2 to 5.


In [ ]:
run(
    "eval-datasets",
    "--models",
    *MODELS,
    "--episodes",
    EVAL_EPISODES,
    "--repetitions",
    REPETITIONS,
    "--seed",
    SEED,
)
for group in ("sources", "loads", "congestions"):
    for metric in ("fct_gain", "power_saving", "efficiency"):
        show_png(f"eval/datasets_{group}_{metric}.png")

## Evaluation 6 — enabled controllable switches

Keep V0 running. Each model and the all-on baseline run a single episode with the same seed. The plot shows the total number of enabled support bits across their action masks after every joint decision, and the legend includes the final switch count. `--dataset` can select another prepared test dataset. Runs extending beyond the dataset include idle time after its traffic ends.


In [ ]:
SWITCH_STEPS = MIN_DECISIONS
run(
    "eval-switches",
    "--network",
    "v0.json",
    "--steps",
    SWITCH_STEPS,
    "--models",
    *MODELS,
    "--seed",
    SEED,
    "--dataset",
    "dc_t1",
)
show_png("eval/switches_v0.png")

## Evaluations 1 and 4 — collect one topology at a time

Set `CURRENT_TOPOLOGY` to the topology running outside the notebook, then execute the collection cell. Repeat this section for V0 through V4, restarting both external services between topologies. V4 automatically uses its larger-host test datasets.

Evaluation 1 collects 16 all-on DC T0 embeddings per topology. Evaluation 4 evaluates the three frozen models and their all-on baseline on synthetic DC T0. Its reward CSVs average raw episode returns across repetitions.


In [ ]:
CURRENT_TOPOLOGY = "v0"  # Change to v1, v2, v3, and v4 after external restarts.
assert CURRENT_TOPOLOGY in {f"v{i}" for i in range(5)}
print("Expected running network", ROOT / f"networks/{CURRENT_TOPOLOGY}.json")

In [ ]:
run(
    "eval-topology-embeddings",
    "--network",
    f"{CURRENT_TOPOLOGY}.json",
    "--model",
    MODELS[0],
    "--seed",
    SEED,
)
run(
    "eval-topologies",
    "--network",
    f"{CURRENT_TOPOLOGY}.json",
    "--models",
    *MODELS,
    "--episodes",
    EVAL_EPISODES,
    "--repetitions",
    REPETITIONS,
    "--seed",
    SEED,
)
for label in MODELS:
    show_png(f"eval/{label}_{CURRENT_TOPOLOGY}.png")

## Inspect aggregate topology figures

After collecting all five topologies, these commands redraw the topology figures from saved measurements. The topology scatter contains 80 points in five classes. Each metric chart has five topology groups and three model bars per group.


In [ ]:
collected = [
    topology
    for topology in (f"v{i}" for i in range(5))
    if (ROOT / f"eval/topologies_{topology}.json").exists()
    and (ROOT / f"eval/topology_embeddings_{topology}.json").exists()
]
print("Collected", collected)
if len(collected) == 5:
    run(
        "eval-topology-embeddings",
        "--model",
        MODELS[0],
        "--seed",
        SEED,
        "--plot-only",
    )
    run("eval-topologies", "--models", *MODELS, "--plot-only")
    show_png("eval/topology_embeddings.png")
    for metric in ("fct_gain", "power_saving", "efficiency"):
        show_png(f"eval/topologies_{metric}.png")

## Evaluation 5 — pruning reward comparison

Compare cumulative reward over episodes and percentage change relative to the no-pruning run. Curves align by episode number. Relative values use the absolute no-pruning reward as the denominator.


In [ ]:
run("eval-pruning")
show_png("eval/pruning_reward.png")
show_png("eval/pruning_relative.png")

## Inspect saved data

Training returns live beside each inference model. Evaluation 4 saves three reward CSVs per topology under `eval/`. JSON reports retain repetition seeds and raw FCT and power measurements. The drain period is configurable in `settings/default.json`. Incomplete deliveries are ignored afterwards.


In [ ]:
import csv

with (ROOT / "models/dc_t0.csv").open() as stream:
    rows = list(csv.DictReader(stream))
print("First training episodes", rows[:5])

report = json.loads((ROOT / "eval/datasets.json").read_text())
print("Compared models", report["models"])
print("Evaluated datasets", [row["dataset"] for row in report["conditions"]])